# DMS Data Extraction Toolkit from *mzml* files

*Created by Thao NT on Thurs July 31 2025, last edit on Aug 11, 2026*

This processing script only works for .mzML files converted from .wiff files by MSConvert (ProteoWizard).
This script will process DMS datafiles for each lipid located in folders specified by users.

## Setup packages and libraries

In [ ]:
# Install required packages
required = {'pandas', 'numpy', 'ipywidgets', 'io', 'pyopenms'}

%pip install required


In [ ]:
# Import basic system parameters and functions
import sys
import os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from ipywidgets import Output
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
import io
import pyopenms as oms

currentpath = os.getcwd()
os.chdir(currentpath)

## Obtain important information and file location

In [ ]:
# Prompt Users to enter CoV range
input_CoV = widgets.Combobox(
    placeholder= "Start CoV, Stop CoV, Step CoV",
    description = "Please enter:",
    disabled=False)
display(input_CoV)

In [ ]:
# Access entered CoV input
entered_CoV = input_CoV.value

params = [item.strip() for item in entered_CoV.split(',')]
CoV_params = [float(s) for s in params]
print(CoV_params)

# Compute full range of CoV and count
StartCoV = CoV_params[0]
StopCoV = CoV_params[1]
StepCoV = CoV_params[2]

CoV_range = np.arange(StartCoV, StopCoV+StepCoV, StepCoV)
np.around(CoV_range, decimals=1)

In [ ]:
# Promp Users to navigate to folder where DMS files are stored

display(HTML('<script>alert("Browse to the folder where DMS files are stored.");</script>'))

InputDir = FileChooser()
display(InputDir)

In [ ]:
if InputDir:
    file_Dir = InputDir.selected
else:
    print("Dear User, you did not choose a folder. I will stop now. DMS Data Extraction Toolkit is exiting.")
    exit()

In [ ]:
# Promp Users to upload DataInfo.csv

display(HTML('<script>alert("Use the Select button to upload DataInfo.csv file.");</script>'))

InputFile = FileChooser()
display(InputFile)

In [ ]:
DataInfoFile = pd.read_csv(InputFile.selected, sep=',', header=0)
                           

## FileCompiler function

In [ ]:
# i. Define Function FileCompiler to extract intensity from each SV

def FileCompiler (file_Dir):
    df_total = pd.DataFrame()
    for dirnames in os.listdir(file_Dir):
        subfolderpath = os.path.join(file_Dir, dirnames)
        filtered_DataInfoFile = DataInfoFile[DataInfoFile['Files Directory'].str.contains(dirnames, na=False)]
        list_DataInfoFile = filtered_DataInfoFile.to_numpy().flatten().tolist()
            
        if list_DataInfoFile:
            isomer = list_DataInfoFile[1]
            sphingoidBackbone = list_DataInfoFile[2]
            chainLength = list_DataInfoFile[3]
            unsatUnit = list_DataInfoFile[4]
                
        for filename in os.listdir(subfolderpath):
            if filename.endswith(".mzML") and "SV" in filename:
                filepath = os.path.join(subfolderpath, filename)        
                SV_list = []
                df_temp  = pd.DataFrame()
                
                exp = oms.MSExperiment()
                oms.MzMLFile().load(filepath, exp)

                chromatogram_data = exp.getChromatogram(0).get_peaks()
                scanNum = chromatogram_data[0]
                Intensity = chromatogram_data[1]
                
                #the size of chromatogram_data has to equal the range of CoV
                if len(scanNum) == len(CoV_range):
                    print("The number of scans in your datafile matches the CoV range. Proceed.")
                    for i in range(len(CoV_range)):        
                        SV_split1, SV_split2 = filename.split("_",1)
                        SV_list.append(float(SV_split2.split("_",1)[0]))
                else:
                    print("The number of scans in your datafile DOES NOT match the CoV range. Please recheck your data and input information for CoV.")
                    sys.exit()
                
                df_temp['COV'] = CoV_range
                df_temp['Intensity'] = Intensity
                df_temp['SV'] = SV_list
                df_temp['chainLength'] = chainLength
                df_temp['unsaturationUnit'] = unsatUnit
                df_temp['isomer'] = isomer
                df_temp['sphingoidBackbone'] = sphingoidBackbone
                df_temp['lipidSpecies'] = df_temp['chainLength'].astype(str) + ':' + df_temp['unsaturationUnit'].astype(str)
                
                df_total = pd.concat([df_total, df_temp])
    return df_total

# ii. Execute Compiling Function
df_new = pd.DataFrame()
df_new = pd.concat([FileCompiler(file_Dir), df_new])

## Generate Output file

In [ ]:
# Saving file
display(HTML('<script>alert("Please select folder directory to save processed DMS data.");</script>'))

chooser = FileChooser()
display(chooser)

In [ ]:
# Prompt user for saved filename
save_as = widgets.Combobox(
    placeholder="processed DMS data name",
    description = "Save as:",
    disabled=False)
display(save_as)

In [ ]:
save_folder = chooser.selected
save_name = save_as.value

save_path = save_folder + save_name + ".csv"

df_new.to_csv(save_path, index=False)
print(f"DataFrame saved to: {save_path}")


In [ ]:
pip list